In [0]:
import requests
import uuid
import hashlib

from datetime import datetime, timezone, timedelta

pipeline_start_timestamp = datetime.now(timezone.utc)

PIPELINE_NAME = "bcb_sgs_selic"

print("Pipeline start:", pipeline_start_timestamp)

In [0]:
BCB_BASE_URL = "https://api.bcb.gov.br/dados/serie"
SOURCE_SYSTEM = "BCB_SGS"
SERIES_CODE = "432"
LOOKBACK_DAYS = 30
REQUEST_TIMEOUT_SECONDS = 30

In [0]:
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

start_date_bcb = start_date.strftime("%d/%m/%Y")
end_date_bcb = end_date.strftime("%d/%m/%Y")

url = f"{BCB_BASE_URL}/bcdata.sgs.{SERIES_CODE}/dados"

params = {
    "formato": "json",
    "dataInicial": start_date_bcb,
    "dataFinal": end_date_bcb
}

response = requests.get(
    url,
    params=params,
    timeout=REQUEST_TIMEOUT_SECONDS
)

response.raise_for_status()

records = response.json()

print("Source:", SOURCE_SYSTEM)
print("Series:", SERIES_CODE)
print("Start date:", start_date_bcb)
print("End date:", end_date_bcb)
print("HTTP Status:", response.status_code)
print("URL:", response.url)
print("Records received:", len(records))
print("First record:", records[0] if records else None)

In [0]:
if not records:
    raise ValueError(
        f"No records returned by BCB SGS series {SERIES_CODE} "
        f"between {start_date_bcb} and {end_date_bcb}"
    )

execution_id = str(uuid.uuid4())
ingestion_timestamp = datetime.now(timezone.utc)

bronze_records = [
    (
        item["data"],
        item["valor"],
        SOURCE_SYSTEM,
        SERIES_CODE,
        ingestion_timestamp,
        ingestion_timestamp.date(),
        execution_id,
        hashlib.sha256(
            f"{SERIES_CODE}||{item['data']}||{item['valor']}".encode("utf-8")
        ).hexdigest()
    )
    for item in records
]

columns = [
    "reference_date_raw",
    "value_raw",
    "source_system",
    "source_series_code",
    "ingestion_timestamp",
    "ingestion_date",
    "execution_id",
    "record_hash"
]

df_bronze = spark.createDataFrame(
    bronze_records,
    schema=columns
)

df_bronze.createOrReplaceTempView(
    "vw_bcb_sgs_selic_bronze"
)

print("Execution ID:", execution_id)
print("Records prepared:", len(bronze_records))

In [0]:
bronze_count_before = spark.sql("""
    SELECT COUNT(*) AS count
    FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
""").collect()[0]["count"]

print("Bronze count before:", bronze_count_before)

In [0]:
%sql
MERGE INTO workspace.brazilian_economic_bronze.bcb_sgs_selic AS target

USING vw_bcb_sgs_selic_bronze AS source

ON target.record_hash = source.record_hash

WHEN NOT MATCHED THEN INSERT (
    reference_date_raw,
    value_raw,
    source_system,
    source_series_code,
    ingestion_timestamp,
    ingestion_date,
    execution_id,
    record_hash
)
VALUES (
    source.reference_date_raw,
    source.value_raw,
    source.source_system,
    source.source_series_code,
    source.ingestion_timestamp,
    source.ingestion_date,
    source.execution_id,
    source.record_hash
);

In [0]:
bronze_count_after = spark.sql("""
    SELECT COUNT(*) AS count
    FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
""").collect()[0]["count"]

records_inserted_bronze = bronze_count_after - bronze_count_before

print("Records received:", len(records))
print("Bronze count before:", bronze_count_before)
print("Bronze count after:", bronze_count_after)
print("Records inserted into Bronze:", records_inserted_bronze)

In [0]:
%sql
SELECT COUNT(*) AS record_count
FROM workspace.brazilian_economic_bronze.bcb_sgs_selic;

In [0]:
%sql
SELECT
    reference_date_raw,
    value_raw,
    source_system,
    source_series_code,
    ingestion_timestamp,
    ingestion_date,
    execution_id
FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
ORDER BY reference_date_raw
LIMIT 5;

In [0]:
%sql
SELECT
    execution_id,
    COUNT(*) AS records_ingested
FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
GROUP BY execution_id;